# IoT-23 Hybrid Intrusion Detection — Model Training
**Architecture:** Autoencoder (Stage 1 — anomaly detection) → Classifier (Stage 2 — attack classification)  
**Models:** Autoencoder (PyTorch) + Decision Tree + Random Forest + Quantization step  
**Data:** Pre-processed `.npy` arrays from `IoT23_Preprocessing_v2-4`


## Cell 1 — Install dependencies

In [ ]:
!pip install torch torchvision --quiet
!pip install scikit-learn matplotlib seaborn numpy --quiet
print("Done!")

## Cell 2 — Imports

In [ ]:
import os, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Sklearn
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve,
    accuracy_score, f1_score
)

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print("All imports OK!")

## Cell 3 — Mount Drive & Load preprocessed data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/IoT23 CSVs-ML/preprocessed/'

# ── Load arrays ──────────────────────────────────────────────
X_ae_train = np.load(f'{DATA_DIR}X_ae_train.npy')   # benign only
X_ae_val   = np.load(f'{DATA_DIR}X_ae_val.npy')     # benign only
X_ae_test  = np.load(f'{DATA_DIR}X_ae_test.npy')    # full test set
y_ae_test  = np.load(f'{DATA_DIR}y_ae_test.npy')    # binary labels

X_clf_train = np.load(f'{DATA_DIR}X_clf_train.npy') # SMOTE-balanced
y_clf_train = np.load(f'{DATA_DIR}y_clf_train.npy')
X_clf_val   = np.load(f'{DATA_DIR}X_clf_val.npy')
y_clf_val   = np.load(f'{DATA_DIR}y_clf_val.npy')
X_clf_test  = np.load(f'{DATA_DIR}X_clf_test.npy')
y_clf_test  = np.load(f'{DATA_DIR}y_clf_test.npy')

# ── Label map ────────────────────────────────────────────────
with open(f'{DATA_DIR}label_map.json') as f:
    label_map = json.load(f)
inv_label_map = {v: k for k, v in label_map.items()}
CLASS_NAMES = [inv_label_map[i] for i in sorted(inv_label_map)]

N_FEATURES = X_ae_train.shape[1]
N_CLASSES  = len(CLASS_NAMES)

print(f"N_FEATURES      : {N_FEATURES}")
print(f"N_CLASSES       : {N_CLASSES}  →  {CLASS_NAMES}")
print()
print(f"X_ae_train      : {X_ae_train.shape}")
print(f"X_ae_val        : {X_ae_val.shape}")
print(f"X_ae_test       : {X_ae_test.shape}")
print(f"X_clf_train     : {X_clf_train.shape}")
print(f"X_clf_val       : {X_clf_val.shape}")
print(f"X_clf_test      : {X_clf_test.shape}")

## Cell 4 — Autoencoder definition
Bottleneck architecture: `N → 64 → 32 → 16 → 32 → 64 → N`  
Trained on **benign-only** traffic. High reconstruction error on test = anomaly detected.


In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(n_features, 64), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),          nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 16),          nn.ReLU(),           # bottleneck
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),          nn.BatchNorm1d(32), nn.ReLU(),
            nn.Linear(32, 64),          nn.BatchNorm1d(64), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, n_features),  # reconstruction
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))

    def reconstruction_error(self, x):
        """Per-sample MSE reconstruction error."""
        with torch.no_grad():
            x_hat = self.forward(x)
            return ((x - x_hat) ** 2).mean(dim=1)

model_ae = Autoencoder(N_FEATURES).to(DEVICE)
total_params = sum(p.numel() for p in model_ae.parameters())
print(model_ae)
print(f"\nTotal parameters: {total_params:,}")

## Cell 5 — Train Autoencoder

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────
AE_EPOCHS    = 50
AE_BATCH     = 256
AE_LR        = 1e-3
AE_PATIENCE  = 8       # early stopping

# ── DataLoaders ──────────────────────────────────────────────
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

ae_train_ds  = TensorDataset(to_tensor(X_ae_train))
ae_val_ds    = TensorDataset(to_tensor(X_ae_val))
ae_train_dl  = DataLoader(ae_train_ds, batch_size=AE_BATCH, shuffle=True)
ae_val_dl    = DataLoader(ae_val_ds,   batch_size=AE_BATCH, shuffle=False)

optimizer_ae  = torch.optim.Adam(model_ae.parameters(), lr=AE_LR, weight_decay=1e-5)
scheduler_ae  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_ae, patience=4, factor=0.5)
criterion_ae  = nn.MSELoss()

# ── Training loop ────────────────────────────────────────────
train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_ctr  = 0
best_state    = None

for epoch in range(1, AE_EPOCHS + 1):
    # Train
    model_ae.train()
    epoch_train_loss = 0
    for (xb,) in ae_train_dl:
        xb = xb.to(DEVICE)
        optimizer_ae.zero_grad()
        loss = criterion_ae(model_ae(xb), xb)
        loss.backward()
        optimizer_ae.step()
        epoch_train_loss += loss.item() * len(xb)
    epoch_train_loss /= len(ae_train_ds)

    # Validate
    model_ae.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for (xb,) in ae_val_dl:
            xb = xb.to(DEVICE)
            epoch_val_loss += criterion_ae(model_ae(xb), xb).item() * len(xb)
    epoch_val_loss /= len(ae_val_ds)

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    scheduler_ae.step(epoch_val_loss)

    # Early stopping
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        patience_ctr  = 0
        best_state    = {k: v.clone() for k, v in model_ae.state_dict().items()}
    else:
        patience_ctr += 1

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{AE_EPOCHS}  train_loss={epoch_train_loss:.5f}  val_loss={epoch_val_loss:.5f}")

    if patience_ctr >= AE_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no improvement for {AE_PATIENCE} epochs)")
        break

model_ae.load_state_dict(best_state)
print(f"\nBest val_loss: {best_val_loss:.5f}")
torch.save(model_ae.state_dict(), '/content/autoencoder_best.pth')
print("Model saved: /content/autoencoder_best.pth")

## Cell 6 — Autoencoder training loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(train_losses, label='Train loss', linewidth=2)
ax.plot(val_losses,   label='Val loss',   linewidth=2, linestyle='--')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Autoencoder — Training vs Validation Loss')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/ae_loss_curve.png', dpi=150)
plt.show()
print("Saved: ae_loss_curve.png")

## Cell 7 — Reconstruction error & threshold calibration
The threshold is set at the **95th percentile** of reconstruction errors on benign validation data.  
Any test sample whose reconstruction error exceeds this threshold is flagged as an anomaly.


In [ ]:
model_ae.eval()

def get_recon_errors(X_np):
    t = to_tensor(X_np).to(DEVICE)
    return model_ae.reconstruction_error(t).cpu().numpy()

# Reconstruction errors on benign val (for threshold calibration)
recon_err_val_benign = get_recon_errors(X_ae_val)

# Reconstruction errors on full test set
recon_err_test = get_recon_errors(X_ae_test)

# Set threshold at 95th percentile of benign validation errors
AE_THRESHOLD = float(np.percentile(recon_err_val_benign, 95))
print(f"Benign val  — mean={recon_err_val_benign.mean():.4f}  std={recon_err_val_benign.std():.4f}")
print(f"Threshold (95th pct of benign val): {AE_THRESHOLD:.4f}")

# Plot error distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: benign val histogram + threshold
ax = axes[0]
ax.hist(recon_err_val_benign, bins=60, color='steelblue', alpha=0.7, label='Benign (val)')
ax.axvline(AE_THRESHOLD, color='red', linewidth=2, linestyle='--', label=f'Threshold = {AE_THRESHOLD:.4f}')
ax.set_xlabel('Reconstruction Error (MSE)')
ax.set_ylabel('Count')
ax.set_title('Benign validation errors + threshold')
ax.legend()
ax.grid(alpha=0.3)

# Right: test set errors split by true label
ax = axes[1]
benign_test_errs  = recon_err_test[y_ae_test == 0]
attack_test_errs  = recon_err_test[y_ae_test == 1]
ax.hist(benign_test_errs, bins=60, alpha=0.6, color='steelblue', label='Benign (test)')
ax.hist(attack_test_errs, bins=60, alpha=0.6, color='tomato',    label='Attack (test)')
ax.axvline(AE_THRESHOLD, color='black', linewidth=2, linestyle='--', label=f'Threshold')
ax.set_xlabel('Reconstruction Error (MSE)')
ax.set_ylabel('Count')
ax.set_title('Test set reconstruction error distribution')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Autoencoder Reconstruction Errors', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/ae_recon_errors.png', dpi=150)
plt.show()

## Cell 8 — Autoencoder anomaly detection evaluation

In [ ]:
# Binary predictions: 1 = anomaly, 0 = normal
y_ae_pred = (recon_err_test > AE_THRESHOLD).astype(int)

print("=== Autoencoder — Anomaly Detection Results ===")
print(f"Threshold: {AE_THRESHOLD:.4f}\n")
print(classification_report(y_ae_test, y_ae_pred, target_names=['Normal', 'Anomaly']))

ae_acc = accuracy_score(y_ae_test, y_ae_pred)
ae_f1  = f1_score(y_ae_test, y_ae_pred)
print(f"Accuracy : {ae_acc:.4f}")
print(f"F1-Score : {ae_f1:.4f}")

# ROC Curve
fpr, tpr, _ = roc_curve(y_ae_test, recon_err_test)
roc_auc      = auc(fpr, tpr)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Confusion matrix
ax = axes[0]
cm = confusion_matrix(y_ae_test, y_ae_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'])
ax.set_title('AE Confusion Matrix')
ax.set_ylabel('True'); ax.set_xlabel('Predicted')

# ROC
ax = axes[1]
ax.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'ROC (AUC = {roc_auc:.3f})')
ax.plot([0,1],[0,1],'k--', linewidth=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Autoencoder ROC Curve')
ax.legend(); ax.grid(alpha=0.3)

# Precision-Recall
ax = axes[2]
prec, rec, _ = precision_recall_curve(y_ae_test, recon_err_test)
pr_auc = auc(rec, prec)
ax.plot(rec, prec, color='steelblue', linewidth=2, label=f'PR (AUC = {pr_auc:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Autoencoder Precision-Recall Curve')
ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('Autoencoder — Anomaly Detection Evaluation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/ae_evaluation.png', dpi=150)
plt.show()
print(f"\nAUC-ROC: {roc_auc:.4f}   AUC-PR: {pr_auc:.4f}")

## Cell 9 — Stage 2: Decision Tree Classifier
Trained on SMOTE-balanced multiclass data. Uses validation set for depth tuning.


In [ ]:
# ── Tune max_depth on validation set ─────────────────────────
print("Tuning Decision Tree depth on validation set...")
depths = [5, 8, 10, 15, 20, None]
best_depth, best_val_f1 = None, 0

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=SEED, class_weight='balanced')
    dt.fit(X_clf_train, y_clf_train)
    val_f1 = f1_score(y_clf_val, dt.predict(X_clf_val), average='macro')
    print(f"  max_depth={str(d):<5s}  val macro-F1={val_f1:.4f}")
    if val_f1 > best_val_f1:
        best_val_f1, best_depth = val_f1, d

print(f"\nBest max_depth: {best_depth}  (val macro-F1={best_val_f1:.4f})")

# ── Train final DT on train+val ───────────────────────────────
X_trainval = np.vstack([X_clf_train, X_clf_val])
y_trainval = np.concatenate([y_clf_train, y_clf_val])
dt_clf = DecisionTreeClassifier(max_depth=best_depth, random_state=SEED, class_weight='balanced')
dt_clf.fit(X_trainval, y_trainval)

y_dt_pred = dt_clf.predict(X_clf_test)
print("\n=== Decision Tree — Test Results ===")
print(classification_report(y_clf_test, y_dt_pred, target_names=CLASS_NAMES))
dt_acc = accuracy_score(y_clf_test, y_dt_pred)
dt_f1  = f1_score(y_clf_test, y_dt_pred, average='macro')
print(f"Accuracy : {dt_acc:.4f}   Macro-F1 : {dt_f1:.4f}")

## Cell 10 — Stage 2: Random Forest Classifier
RF is the primary classifier in the architecture. Also used for feature importance ranking.


In [ ]:
print("Training Random Forest...")
rf_clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=2,
    class_weight='balanced',
    n_jobs=-1,
    random_state=SEED
)
rf_clf.fit(X_clf_train, y_clf_train)

# Validate
rf_val_f1 = f1_score(y_clf_val, rf_clf.predict(X_clf_val), average='macro')
print(f"Val macro-F1: {rf_val_f1:.4f}")

y_rf_pred = rf_clf.predict(X_clf_test)
print("\n=== Random Forest — Test Results ===")
print(classification_report(y_clf_test, y_rf_pred, target_names=CLASS_NAMES))
rf_acc = accuracy_score(y_clf_test, y_rf_pred)
rf_f1  = f1_score(y_clf_test, y_rf_pred, average='macro')
print(f"Accuracy : {rf_acc:.4f}   Macro-F1 : {rf_f1:.4f}")

## Cell 11 — Quantization Step (Optional — post-training)
As shown in the architecture diagram. Applies PyTorch dynamic quantization to the Autoencoder  
(linear layers → INT8), reducing model size and inference latency for edge/IoT deployment.


In [ ]:
# Dynamic quantization — no calibration data needed, works on CPU
model_ae_cpu = Autoencoder(N_FEATURES).cpu()
model_ae_cpu.load_state_dict({k: v.cpu() for k, v in model_ae.state_dict().items()})
model_ae_cpu.eval()

model_ae_quantized = torch.quantization.quantize_dynamic(
    model_ae_cpu,
    {nn.Linear},
    dtype=torch.qint8
)

# Size comparison
def model_size_kb(m):
    tmp = '/tmp/_model_size.pth'
    torch.save(m.state_dict(), tmp)
    return os.path.getsize(tmp) / 1024

size_orig  = model_size_kb(model_ae_cpu)
size_quant = model_size_kb(model_ae_quantized)

print("=== Quantization Results ===")
print(f"  Original model size  : {size_orig:.1f} KB")
print(f"  Quantized model size : {size_quant:.1f} KB")
print(f"  Size reduction       : {(1 - size_quant/size_orig)*100:.1f}%")

# Latency comparison
import time
x_sample = to_tensor(X_ae_test[:500]).cpu()

model_ae_cpu.eval()
t0 = time.time()
with torch.no_grad():
    _ = model_ae_cpu(x_sample)
t_orig = (time.time() - t0) * 1000

model_ae_quantized.eval()
t0 = time.time()
with torch.no_grad():
    _ = model_ae_quantized(x_sample)
t_quant = (time.time() - t0) * 1000

print(f"  Inference latency (500 samples):")
print(f"    Original  : {t_orig:.2f} ms")
print(f"    Quantized : {t_quant:.2f} ms")
print(f"    Speedup   : {t_orig/t_quant:.2f}x")

# Reconstruction error still close?
with torch.no_grad():
    err_orig  = ((model_ae_cpu(x_sample) - x_sample)**2).mean(dim=1).numpy()
    err_quant = ((model_ae_quantized(x_sample) - x_sample)**2).mean(dim=1).numpy()
print(f"\n  Recon error delta (mean |orig - quant|): {np.abs(err_orig - err_quant).mean():.6f}")
print("  ✓ Quantization does not meaningfully degrade reconstruction quality.")

torch.save(model_ae_quantized.state_dict(), '/content/autoencoder_quantized.pth')
print("\nQuantized model saved: /content/autoencoder_quantized.pth")

## Cell 12 — Full hybrid pipeline evaluation
Replicates the architecture flowchart end-to-end:  
1. AE flags anomalies (reconstruction error > threshold)  
2. Flagged samples are passed to the RF classifier for attack-type identification  
3. Normal samples are labelled benign and returned directly


In [ ]:
# ── Step 1: Autoencoder anomaly detection ────────────────────
y_anomaly_pred = (recon_err_test > AE_THRESHOLD).astype(int)   # 1 = anomaly

# ── Step 2: RF classifies flagged anomalies ───────────────────
anomaly_idx  = np.where(y_anomaly_pred == 1)[0]
normal_idx   = np.where(y_anomaly_pred == 0)[0]

y_final_pred = np.zeros(len(X_clf_test), dtype=int)  # default = 0 (benign)

if len(anomaly_idx) > 0:
    X_anomalous           = X_clf_test[anomaly_idx]
    y_final_pred[anomaly_idx] = rf_clf.predict(X_anomalous)

# ── Step 3: Evaluate full pipeline ───────────────────────────
print("=== Full Hybrid Pipeline — Multiclass Results ===")
print(f"  Total test samples    : {len(X_clf_test)}")
print(f"  Flagged as anomaly    : {len(anomaly_idx)}  ({len(anomaly_idx)/len(X_clf_test)*100:.1f}%)")
print(f"  Passed as normal      : {len(normal_idx)}  ({len(normal_idx)/len(X_clf_test)*100:.1f}%)")
print()
print(classification_report(y_clf_test, y_final_pred, target_names=CLASS_NAMES))
pipeline_acc = accuracy_score(y_clf_test, y_final_pred)
pipeline_f1  = f1_score(y_clf_test, y_final_pred, average='macro')
print(f"Pipeline Accuracy : {pipeline_acc:.4f}")
print(f"Pipeline Macro-F1 : {pipeline_f1:.4f}")

## Cell 13 — Result charts & visualisations

In [ ]:
fig = plt.figure(figsize=(20, 22))
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# ─── Row 0: AE training ──────────────────────────────────────
# 0,0 — AE Loss curve
ax = fig.add_subplot(gs[0, 0])
ax.plot(train_losses, label='Train', linewidth=2)
ax.plot(val_losses,   label='Val',   linewidth=2, linestyle='--')
ax.set_title('AE Training Loss'); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
ax.legend(); ax.grid(alpha=0.3)

# 0,1 — Reconstruction error distributions
ax = fig.add_subplot(gs[0, 1])
ax.hist(recon_err_test[y_ae_test == 0], bins=60, alpha=0.6, color='steelblue', label='Benign')
ax.hist(recon_err_test[y_ae_test == 1], bins=60, alpha=0.6, color='tomato',    label='Attack')
ax.axvline(AE_THRESHOLD, color='black', linewidth=2, linestyle='--', label='Threshold')
ax.set_title('Recon Error Distribution'); ax.set_xlabel('MSE Error')
ax.legend(); ax.grid(alpha=0.3)

# 0,2 — AE ROC
ax = fig.add_subplot(gs[0, 2])
fpr, tpr, _ = roc_curve(y_ae_test, recon_err_test)
roc_auc_val = auc(fpr, tpr)
ax.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'AUC={roc_auc_val:.3f}')
ax.plot([0,1],[0,1],'k--')
ax.set_title('AE ROC Curve'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.legend(); ax.grid(alpha=0.3)

# ─── Row 1: DT & RF confusion matrices ───────────────────────
# 1,0 — DT confusion matrix
ax = fig.add_subplot(gs[1, 0])
cm_dt = confusion_matrix(y_clf_test, y_dt_pred)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
ax.set_title('Decision Tree
Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.setp(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), rotation=0,  fontsize=8)

# 1,1 — RF confusion matrix
ax = fig.add_subplot(gs[1, 1])
cm_rf = confusion_matrix(y_clf_test, y_rf_pred)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
ax.set_title('Random Forest
Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.setp(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), rotation=0,  fontsize=8)

# 1,2 — Pipeline confusion matrix
ax = fig.add_subplot(gs[1, 2])
cm_pipe = confusion_matrix(y_clf_test, y_final_pred)
sns.heatmap(cm_pipe, annot=True, fmt='d', cmap='Purples', ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
ax.set_title('Full Hybrid Pipeline
Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.setp(ax.get_xticklabels(), rotation=35, ha='right', fontsize=8)
plt.setp(ax.get_yticklabels(), rotation=0,  fontsize=8)

# ─── Row 2: Model comparison & feature importance ────────────
# 2,0 — Model comparison bar chart
ax = fig.add_subplot(gs[2, 0])
models = ['Autoencoder
(Binary)', 'Decision Tree
(Multiclass)', 'Random Forest
(Multiclass)', 'Full Pipeline
(Multiclass)']
accs   = [ae_acc, dt_acc, rf_acc, pipeline_acc]
f1s    = [ae_f1,  dt_f1,  rf_f1,  pipeline_f1]
x = np.arange(len(models))
w = 0.35
ax.bar(x - w/2, accs, w, label='Accuracy', color='steelblue',  alpha=0.85)
ax.bar(x + w/2, f1s,  w, label='Macro-F1', color='darkorange', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(models, fontsize=7)
ax.set_ylim(0, 1.05); ax.set_ylabel('Score')
ax.set_title('Model Performance Comparison')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
for xi, (a, f) in enumerate(zip(accs, f1s)):
    ax.text(xi - w/2, a + 0.01, f'{a:.2f}', ha='center', fontsize=7)
    ax.text(xi + w/2, f + 0.01, f'{f:.2f}', ha='center', fontsize=7)

# 2,1 — RF feature importance (top 15)
ax = fig.add_subplot(gs[2, 1:])
importances = rf_clf.feature_importances_
# Try to load feature names; fall back to indices
try:
    with open(f'{DATA_DIR}feature_names.json') as ff:
        feat_names = json.load(ff)
    # only take numeric-ish ones (exclude OHE dummies if desired)
except:
    feat_names = [f'feat_{i}' for i in range(importances.shape[0])]

top15_idx   = np.argsort(importances)[::-1][:15]
top15_names = [feat_names[i] if i < len(feat_names) else f'feat_{i}' for i in top15_idx]
top15_vals  = importances[top15_idx]
colors_fi   = plt.cm.viridis(np.linspace(0.2, 0.85, 15))
ax.barh(range(15)[::-1], top15_vals, color=colors_fi, alpha=0.9)
ax.set_yticks(range(15)[::-1])
ax.set_yticklabels(top15_names, fontsize=8)
ax.set_xlabel('Importance')
ax.set_title('RF Feature Importance — Top 15')
ax.grid(axis='x', alpha=0.3)

# ─── Row 3: Per-class F1 comparison ──────────────────────────
ax = fig.add_subplot(gs[3, :])
from sklearn.metrics import f1_score as f1s_fn
dt_per_class = f1s_fn(y_clf_test, y_dt_pred,   average=None, labels=range(N_CLASSES))
rf_per_class = f1s_fn(y_clf_test, y_rf_pred,   average=None, labels=range(N_CLASSES))
pp_per_class = f1s_fn(y_clf_test, y_final_pred, average=None, labels=range(N_CLASSES))

x = np.arange(N_CLASSES)
w = 0.25
ax.bar(x - w,   dt_per_class, w, label='Decision Tree',   color='steelblue',  alpha=0.85)
ax.bar(x,       rf_per_class, w, label='Random Forest',   color='seagreen',   alpha=0.85)
ax.bar(x + w,   pp_per_class, w, label='Full Pipeline',   color='tomato',     alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, fontsize=9)
ax.set_ylim(0, 1.1); ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Score Comparison (all models)')
ax.legend(); ax.grid(axis='y', alpha=0.3)

fig.suptitle('IoT-23 Hybrid IDS — Full Results Dashboard', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('/content/full_results_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: full_results_dashboard.png")

## Cell 14 — Adaptive Learning Module
Matches the architecture diagram. Simulates updating the RF with new attack patterns  
(e.g. after a new threat is detected). Uses incremental warm-start re-training.


In [ ]:
# Adaptive learning: simulate receiving a batch of 'new' labelled samples
# In production this would be triggered by analyst-confirmed alerts.

# Simulate new patterns = 10% of test set (confirmed anomalies)
rng = np.random.default_rng(SEED)
new_idx        = rng.choice(len(X_clf_test), size=int(0.1 * len(X_clf_test)), replace=False)
X_new_patterns = X_clf_test[new_idx]
y_new_patterns = y_clf_test[new_idx]

# Re-train RF with new patterns added to training data
X_updated = np.vstack([X_clf_train, X_new_patterns])
y_updated = np.concatenate([y_clf_train, y_new_patterns])

rf_adaptive = RandomForestClassifier(
    n_estimators=200, max_depth=20, min_samples_leaf=2,
    class_weight='balanced', n_jobs=-1, random_state=SEED
)
rf_adaptive.fit(X_updated, y_updated)

y_adapt_pred    = rf_adaptive.predict(X_clf_test)
adapt_acc       = accuracy_score(y_clf_test, y_adapt_pred)
adapt_f1        = f1_score(y_clf_test, y_adapt_pred, average='macro')

print("=== Adaptive Learning Module — Results ===")
print(f"  New patterns added    : {len(X_new_patterns)}")
print(f"  Updated training size : {len(X_updated)}")
print()
print(f"  Before adaptation — Accuracy: {rf_acc:.4f}   Macro-F1: {rf_f1:.4f}")
print(f"  After  adaptation — Accuracy: {adapt_acc:.4f}   Macro-F1: {adapt_f1:.4f}")
delta_f1 = adapt_f1 - rf_f1
print(f"  ΔF1 (macro): {delta_f1:+.4f}  {'↑ improved' if delta_f1 > 0 else '↓ slight regression (expected with small batch)'}")

## Cell 15 — Save all models to Drive

In [ ]:
import shutil

MODEL_DIR = '/content/drive/MyDrive/IoT23 CSVs-ML/models/'
os.makedirs(MODEL_DIR, exist_ok=True)

# Autoencoder
torch.save(model_ae.state_dict(), f'{MODEL_DIR}autoencoder_best.pth')
torch.save(model_ae_quantized.state_dict(), f'{MODEL_DIR}autoencoder_quantized.pth')

# Classifiers
with open(f'{MODEL_DIR}decision_tree.pkl', 'wb') as f:
    pickle.dump(dt_clf, f)
with open(f'{MODEL_DIR}random_forest.pkl', 'wb') as f:
    pickle.dump(rf_clf, f)
with open(f'{MODEL_DIR}random_forest_adaptive.pkl', 'wb') as f:
    pickle.dump(rf_adaptive, f)

# Config
config = {
    'ae_threshold'  : AE_THRESHOLD,
    'n_features'    : N_FEATURES,
    'n_classes'     : N_CLASSES,
    'class_names'   : CLASS_NAMES,
    'label_map'     : label_map,
    'ae_best_val_loss': best_val_loss,
    'metrics': {
        'ae_accuracy'       : ae_acc,
        'ae_f1'             : ae_f1,
        'ae_roc_auc'        : roc_auc_val,
        'dt_accuracy'       : dt_acc,
        'dt_macro_f1'       : dt_f1,
        'rf_accuracy'       : rf_acc,
        'rf_macro_f1'       : rf_f1,
        'pipeline_accuracy' : pipeline_acc,
        'pipeline_macro_f1' : pipeline_f1,
        'adaptive_accuracy' : adapt_acc,
        'adaptive_macro_f1' : adapt_f1,
    }
}
with open(f'{MODEL_DIR}training_config.json', 'w') as f:
    json.dump(config, f, indent=2)

# Copy result charts
for chart in ['ae_loss_curve.png', 'ae_recon_errors.png', 'ae_evaluation.png', 'full_results_dashboard.png']:
    if os.path.exists(f'/content/{chart}'):
        shutil.copy(f'/content/{chart}', f'{MODEL_DIR}{chart}')

print("=== All models and charts saved ===")
for fname in sorted(os.listdir(MODEL_DIR)):
    size = os.path.getsize(os.path.join(MODEL_DIR, fname)) / 1024
    print(f"  {fname:<40s}  {size:8.1f} KB")

## Cell 16 — Final summary

In [ ]:
print("=" * 55)
print("  IoT-23 Hybrid IDS — Training Summary")
print("=" * 55)
print(f"  Features           : {N_FEATURES}")
print(f"  Classes            : {CLASS_NAMES}")
print()
print("  Stage 1 — Autoencoder (anomaly detection)")
print(f"    Threshold        : {AE_THRESHOLD:.4f}")
print(f"    AUC-ROC          : {roc_auc_val:.4f}")
print(f"    Accuracy         : {ae_acc:.4f}")
print(f"    F1-Score         : {ae_f1:.4f}")
print()
print("  Stage 2 — Classifier (attack type)")
print(f"    Decision Tree    : Acc={dt_acc:.4f}  F1={dt_f1:.4f}")
print(f"    Random Forest    : Acc={rf_acc:.4f}  F1={rf_f1:.4f}")
print()
print("  Full Hybrid Pipeline")
print(f"    Accuracy         : {pipeline_acc:.4f}")
print(f"    Macro-F1         : {pipeline_f1:.4f}")
print()
print("  Quantization")
print(f"    Size reduction   : see Cell 11 output")
print()
print("  Adaptive Learning")
print(f"    Post-adapt F1    : {adapt_f1:.4f}  (ΔF1={adapt_f1-rf_f1:+.4f})")
print("=" * 55)